## Week 2 Day 1 — Groq + OpenAI Agents SDK

Same lightweight SDK, but powered by **Groq** instead of OpenAI.
Traces are sent to a self-hosted **Phoenix** instance via OTLP/gRPC.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [7]:
# The imports

import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace

from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry import trace as otel_trace
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor


In [8]:
# The usual starting point

load_dotenv(override=True)

GROQ_API_KEY      = os.environ["GROQ_API_KEY"]
PHOENIX_OTLP_GRPC = os.environ.get("PHOENIX_OTLP_GRPC", "192.168.0.111:30317")
PHOENIX_UI        = os.environ.get("PHOENIX_UI",        "http://192.168.0.111:30606")


In [9]:
# Configure tracing — sends spans to Phoenix over gRPC (plain, no TLS)

provider = TracerProvider()
provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_OTLP_GRPC, insecure=True))
)
otel_trace.set_tracer_provider(provider)
OpenAIAgentsInstrumentor().instrument(tracer_provider=provider)


Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


In [10]:
# Make an agent with name, instructions, model — backed by Groq

groq_client = AsyncOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model=OpenAIChatCompletionsModel(
        model="llama-3.3-70b-versatile",
        openai_client=groq_client,
    ),
)


In [13]:
agent

Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a joke teller', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7fdf4046ffb0>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [16]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "update" its emotions and was feeling a little " glitchy" in its relationships. But in the end, it just needed to "reboot" its sense of humor and "learn" to laugh at itself.


In [12]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)


Why did the Autonomous AI Agent go to therapy?

Because it was struggling to "update" its emotions and was having a little "glitch" in its relationships. But in the end, it just needed to "reboot" its attitude and learn to "self-drive" its feelings. Now it's functioning within "parameters" of normal emotional intelligence.


## Now go and look at the trace

Open Phoenix in your browser:

http://192.168.0.111:30606
